# 00 - Setup e inventario

**Objetivo:** confirmar o ambiente, localizar a base e produzir um inventario legivel dos arquivos telemetricos.

Este notebook **nao treina modelos**. Ele serve para garantir que o projeto parte de arquivos corretos.

### Saidas
- `outputs/tables/inventory_37_stations.csv`
- `outputs/tables/inventory_37_stations.html`
- `outputs/tables/inventory_variable_counts.csv`
- `outputs/tables/package_versions.csv`

> Para tabelas grandes, o notebook mostra apenas uma amostra e salva a tabela completa em `outputs/tables/`.

## Antes de executar

Confirme no canto superior direito do VS Code que o kernel e o `.venv`.

A verificacao definitiva e:
```python
import sys
print(sys.executable)
```
O caminho deve terminar em `.venv\Scripts\python.exe`.

## Relatorios visuais

Os notebooks 05, 07 e 08 agora geram **HTML, PDF e dashboard HTML interativo**.
Se o `plotly` ainda nao estiver instalado, execute uma vez no terminal com `(.venv)`:

```powershell
python -m pip install plotly
```

Ou instale tudo a partir do `requirements.txt` incluido neste pacote.


In [8]:
import sys
print(sys.executable)
assert ".venv" in sys.executable.lower(), (
    "O notebook nao esta usando o .venv. Troque o kernel antes de continuar."
)

c:\Users\osmar\Área de Trabalho\Field-Project\.venv\Scripts\python.exe


In [9]:
from pathlib import Path
import re
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------------
# Localiza a raiz do projeto e o dataset de forma robusta.
# Funciona se o notebook estiver em Field-Project/notebooks.
# ------------------------------------------------------------------
CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD

RAW_PARENT = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"

OUTPUTS = ROOT / "outputs"
FIGURES = OUTPUTS / "figures"
TABLES = OUTPUTS / "tables"
PREDICTIONS = OUTPUTS / "predictions"

for p in [INTERIM, PROCESSED, MODELS, FIGURES, TABLES, PREDICTIONS]:
    p.mkdir(parents=True, exist_ok=True)

def find_dataset_root(raw_parent: Path) -> Path:
    """Procura uma pasta que contenha Data/SHP e Data/Telemetric_Time_Series."""
    candidates = [raw_parent] + [p for p in raw_parent.rglob("*") if p.is_dir()]
    for p in candidates:
        data = p / "Data"
        if (data / "SHP").exists() and (data / "Telemetric_Time_Series").exists():
            return p
    raise FileNotFoundError(
        "Nao encontrei o dataset. Extraia o .rar dentro de data/raw/ "
        "mantendo a pasta Data/ com SHP/, Radar/ e Telemetric_Time_Series/."
    )

RAW = find_dataset_root(RAW_PARENT)
DATA = RAW / "Data"
SHP = DATA / "SHP"
TELE = DATA / "Telemetric_Time_Series"
RADAR = DATA / "Radar"

print("ROOT:", ROOT)
print("Dataset:", RAW)

ROOT: C:\Users\osmar\Área de Trabalho\Field-Project
Dataset: C:\Users\osmar\Área de Trabalho\Field-Project\data\raw\TTI-HydroMet_Dataset_version2


In [10]:
def save_table(df, stem, preview_rows=15, sort_by=None, ascending=True):
    """
    Salva a tabela completa em CSV e HTML e mostra apenas uma amostra no notebook.
    Isso evita outputs gigantes e facilita abrir a tabela fora do Jupyter.
    """
    z = df.copy()
    if sort_by is not None and sort_by in z.columns:
        z = z.sort_values(sort_by, ascending=ascending)

    csv_path = TABLES / f"{stem}.csv"
    html_path = TABLES / f"{stem}.html"

    z.to_csv(csv_path, index=False)
    z.to_html(html_path, index=False)

    display(Markdown(
        f"**Tabela completa salva em:**  \n"
        f"- `{csv_path.relative_to(ROOT)}`  \n"
        f"- `{html_path.relative_to(ROOT)}`  \n"
        f"Mostrando apenas as primeiras {min(preview_rows, len(z))} linhas:"
    ))
    display(z.head(preview_rows))
    return z

def save_figure(fig, stem, dpi=220):
    """Salva uma figura em PNG e a mantem visivel no notebook."""
    path = FIGURES / f"{stem}.png"
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    display(Markdown(f"**Figura salva em:** `{path.relative_to(ROOT)}`"))
    return path

In [11]:
import geopandas as gpd
import sklearn
import xgboost

required = [
    SHP / "TRW.shp",
    SHP / "Tamanduatei_river.shp",
    SHP / "Supplementary_hydrography.shp",
    SHP / "Rainfall_gauge_stations.shp",
    SHP / "413_stage-station.shp",
    TELE / "P_413_INST.csv",
]

check = pd.DataFrame({
    "arquivo": [str(p.relative_to(ROOT)) for p in required],
    "existe": [p.exists() for p in required],
})
display(check)

if not check["existe"].all():
    raise FileNotFoundError("Ha arquivos essenciais ausentes. Veja a tabela acima.")

tele_files = sorted(TELE.glob("P_*_INST.csv"))
print("CSVs telemetricos encontrados:", len(tele_files))
assert len(tele_files) == 37, f"Esperados 37; encontrados {len(tele_files)}."

,arquivo,existe
0,data\raw\TTI-HydroMet_Dataset_version2\Data\SH...,True
1,data\raw\TTI-HydroMet_Dataset_version2\Data\SH...,True
2,data\raw\TTI-HydroMet_Dataset_version2\Data\SH...,True
3,data\raw\TTI-HydroMet_Dataset_version2\Data\SH...,True
4,data\raw\TTI-HydroMet_Dataset_version2\Data\SH...,True
5,data\raw\TTI-HydroMet_Dataset_version2\Data\Te...,True


CSVs telemetricos encontrados: 37


In [12]:
def station_id_from_path(path):
    m = re.search(r"P_(\d+)_INST", path.stem)
    return int(m.group(1)) if m else None

def inspect_station_file(path):
    df = pd.read_csv(path, low_memory=False)
    out = {
        "station": station_id_from_path(path),
        "file": path.name,
        "rows": len(df),
        "n_columns": len(df.columns),
        "columns": ", ".join(df.columns.astype(str)),
    }

    if "DATA" in df.columns:
        dates = pd.to_datetime(df["DATA"], format="mixed", errors="coerce")
        out["first_time"] = dates.min()
        out["last_time"] = dates.max()
        out["invalid_dates"] = int(dates.isna().sum())
        out["duplicate_timestamps"] = int(dates.duplicated().sum())

    for col in ["PLUmm", "FLUm", "Qm3s"]:
        out[f"has_{col}"] = col in df.columns
        if col in df.columns:
            values = pd.to_numeric(df[col], errors="coerce")
            out[f"{col}_observed"] = int(values.notna().sum())
            out[f"{col}_missing"] = int(values.isna().sum())
            out[f"{col}_min"] = values.min()
            out[f"{col}_max"] = values.max()

    return out

inventory = pd.DataFrame(
    [inspect_station_file(p) for p in tele_files]
).sort_values("station").reset_index(drop=True)

save_table(inventory, "inventory_37_stations", preview_rows=12)

**Tabela completa salva em:**  
- `outputs\tables\inventory_37_stations.csv`  
- `outputs\tables\inventory_37_stations.html`  
Mostrando apenas as primeiras 12 linhas:

,station,file,rows,n_columns,columns,first_time,last_time,invalid_dates,duplicate_timestamps,has_PLUmm,...,has_FLUm,has_Qm3s,FLUm_observed,FLUm_missing,FLUm_min,FLUm_max,Qm3s_observed,Qm3s_missing,Qm3s_min,Qm3s_max
0,143,P_143_INST.csv,722713,5,"Posto, DATA, PLUmm, FLUm, BateriaV",2011-10-13 15:00:00,2025-09-16 23:50:00,0,13,True,...,True,False,720962.0,1751.0,733.045,739.157,NaN,NaN,NaN,NaN
1,275,P_275_INST.csv,958901,6,"Posto, DATA, PLUmm, FLUm, Qm3s, BateriaV",2007-05-28 15:51:00,2025-09-17 01:10:00,0,141,True,...,True,True,949876.0,9025.0,723.725,733.105,819063.0,139838.0,0.032995,146.570742
2,279,P_279_INST.csv,921303,6,"Posto, DATA, PLUmm, FLUm, Qm3s, BateriaV",2006-12-05 18:15:00,2025-09-17 02:20:00,0,77,True,...,True,True,914452.0,6851.0,736.800,755.907,0.0,921303.0,NaN,NaN
3,280,P_280_INST.csv,952178,6,"Posto, DATA, PLUmm, FLUm, Qm3s, BateriaV",2007-07-10 19:12:00,2025-09-17 03:40:00,0,122,True,...,True,True,936385.0,15793.0,734.174,740.026,0.0,952178.0,NaN,NaN
4,283,P_283_INST.csv,949598,6,"Posto, DATA, PLUmm, FLUm, Qm3s, BateriaV",2007-08-06 16:30:00,2025-09-17 05:20:00,0,23,True,...,True,True,931601.0,17997.0,1.358,733.473,800841.0,148757.0,0.000000,928.144000
5,413,P_413_INST.csv,791577,5,"Posto, DATA, PLUmm, FLUm, BateriaV",2006-11-01 00:00:00,2025-09-17 06:40:00,0,16,True,...,True,False,781950.0,9627.0,711.103,726.549,NaN,NaN,NaN,NaN
6,511,P_511_INST.csv,680188,4,"Posto, DATA, PLUmm, BateriaV",2006-11-01 00:00:00,2025-09-17 08:10:00,0,13,True,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,563,P_563_INST.csv,607935,5,"Posto, DATA, PLUmm, FLUm, BateriaV",2014-02-06 17:20:00,2025-09-17 09:00:00,0,13,True,...,True,False,606574.0,1361.0,739.475,749.900,NaN,NaN,NaN,NaN
8,629,P_629_INST.csv,553280,5,"Posto, DATA, PLUmm, FLUm, BateriaV",2015-03-06 13:00:00,2025-09-17 09:40:00,0,12,True,...,True,False,551195.0,2085.0,724.354,733.467,NaN,NaN,NaN,NaN
9,730,P_730_INST.csv,273005,4,"Posto, DATA, FLUm, BateriaV",2018-09-06 15:00:00,2025-09-17 10:00:00,0,3,False,...,True,False,268927.0,4078.0,0.000,735.983,NaN,NaN,NaN,NaN


,station,file,rows,n_columns,columns,first_time,last_time,invalid_dates,duplicate_timestamps,has_PLUmm,...,has_FLUm,has_Qm3s,FLUm_observed,FLUm_missing,FLUm_min,FLUm_max,Qm3s_observed,Qm3s_missing,Qm3s_min,Qm3s_max
0,143,P_143_INST.csv,722713,5,"Posto, DATA, PLUmm, FLUm, BateriaV",2011-10-13 15:00:00,2025-09-16 23:50:00,0,13,True,...,True,False,720962.0,1751.0,733.045,739.157,NaN,NaN,NaN,NaN
1,275,P_275_INST.csv,958901,6,"Posto, DATA, PLUmm, FLUm, Qm3s, BateriaV",2007-05-28 15:51:00,2025-09-17 01:10:00,0,141,True,...,True,True,949876.0,9025.0,723.725,733.105,819063.0,139838.0,0.032995,146.570742
2,279,P_279_INST.csv,921303,6,"Posto, DATA, PLUmm, FLUm, Qm3s, BateriaV",2006-12-05 18:15:00,2025-09-17 02:20:00,0,77,True,...,True,True,914452.0,6851.0,736.800,755.907,0.0,921303.0,NaN,NaN
3,280,P_280_INST.csv,952178,6,"Posto, DATA, PLUmm, FLUm, Qm3s, BateriaV",2007-07-10 19:12:00,2025-09-17 03:40:00,0,122,True,...,True,True,936385.0,15793.0,734.174,740.026,0.0,952178.0,NaN,NaN
4,283,P_283_INST.csv,949598,6,"Posto, DATA, PLUmm, FLUm, Qm3s, BateriaV",2007-08-06 16:30:00,2025-09-17 05:20:00,0,23,True,...,True,True,931601.0,17997.0,1.358,733.473,800841.0,148757.0,0.000000,928.144000
5,413,P_413_INST.csv,791577,5,"Posto, DATA, PLUmm, FLUm, BateriaV",2006-11-01 00:00:00,2025-09-17 06:40:00,0,16,True,...,True,False,781950.0,9627.0,711.103,726.549,NaN,NaN,NaN,NaN
6,511,P_511_INST.csv,680188,4,"Posto, DATA, PLUmm, BateriaV",2006-11-01 00:00:00,2025-09-17 08:10:00,0,13,True,...,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,563,P_563_INST.csv,607935,5,"Posto, DATA, PLUmm, FLUm, BateriaV",2014-02-06 17:20:00,2025-09-17 09:00:00,0,13,True,...,True,False,606574.0,1361.0,739.475,749.900,NaN,NaN,NaN,NaN
8,629,P_629_INST.csv,553280,5,"Posto, DATA, PLUmm, FLUm, BateriaV",2015-03-06 13:00:00,2025-09-17 09:40:00,0,12,True,...,True,False,551195.0,2085.0,724.354,733.467,NaN,NaN,NaN,NaN
9,730,P_730_INST.csv,273005,4,"Posto, DATA, FLUm, BateriaV",2018-09-06 15:00:00,2025-09-17 10:00:00,0,3,False,...,True,False,268927.0,4078.0,0.000,735.983,NaN,NaN,NaN,NaN


In [13]:
summary = pd.DataFrame({
    "variavel": ["PLUmm", "FLUm", "Qm3s"],
    "numero_de_estacoes": [
        int(inventory["has_PLUmm"].sum()),
        int(inventory["has_FLUm"].sum()),
        int(inventory["has_Qm3s"].sum()),
    ]
})
save_table(summary, "inventory_variable_counts", preview_rows=10)

**Tabela completa salva em:**  
- `outputs\tables\inventory_variable_counts.csv`  
- `outputs\tables\inventory_variable_counts.html`  
Mostrando apenas as primeiras 3 linhas:

,variavel,numero_de_estacoes
0,PLUmm,28
1,FLUm,29
2,Qm3s,4


,variavel,numero_de_estacoes
0,PLUmm,28
1,FLUm,29
2,Qm3s,4


In [14]:
versions = pd.DataFrame({
    "pacote": ["python", "pandas", "numpy", "matplotlib", "sklearn", "xgboost", "geopandas"],
    "versao": [
        sys.version.split()[0],
        pd.__version__,
        np.__version__,
        __import__("matplotlib").__version__,
        sklearn.__version__,
        xgboost.__version__,
        gpd.__version__,
    ],
})
save_table(versions, "package_versions", preview_rows=20)

**Tabela completa salva em:**  
- `outputs\tables\package_versions.csv`  
- `outputs\tables\package_versions.html`  
Mostrando apenas as primeiras 7 linhas:

,pacote,versao
0,python,3.11.9
1,pandas,3.0.5
2,numpy,2.4.6
3,matplotlib,3.11.1
4,sklearn,1.9.0
5,xgboost,3.2.0
6,geopandas,1.1.4


,pacote,versao
0,python,3.11.9
1,pandas,3.0.5
2,numpy,2.4.6
3,matplotlib,3.11.1
4,sklearn,1.9.0
5,xgboost,3.2.0
6,geopandas,1.1.4
